# NFPP Sodium-Ion ESS Research Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourname/sodium-ion-ess/blob/main/src/report.ipynb)

This notebook implements the complete research pipeline for the Sodium Iron Pyrophosphate (NFPP) battery system.

In [ ]:
import os
import subprocess
import sys

# Google Colab Setup
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('sodium-ion-ess'):
        !git clone https://github.com/yourname/sodium-ion-ess.git
        %cd sodium-ion-ess
    sys.path.append(os.getcwd())

!pip install pybamm scipy matplotlib
import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 1: Cell Verification
Verify the base parameter set against literature-referenced material properties.

In [ ]:
# state: CALIBRATING
from nfpp_sodium_ion.src.calibration.derivation import get_derived_parameters
from nfpp_sodium_ion.src.cell_parameters.cell_alpha import get_parameter_values

derived = get_derived_parameters()
base_params = get_parameter_values()

print("Cell Verification (10Ah design):")
print(f"Target Capacity: {base_params['Nominal cell capacity [A.h]']} Ah")
print(f"Layer Count: {base_params['Number of electrodes connected in parallel to make a cell']}")
print(f"Positive c_max: {derived['c_max_p']:.2f} mol/m3")

## Stage 2: Cell Optimization
Hessian-based design space reduction for optimal performance/cost.

In [ ]:
# state: FILTERING
from src.optimization_pipeline.optimize import NFPPoptimizer
optimizer = NFPPoptimizer()
optimal_theta = optimizer.run_optimization()
print("Optimization Result (L_c, L_a, eps_c, eps_a, r_p):", optimal_theta)

## Stage 3: Stability Validation
Physics consistency check and parameter merging.

In [ ]:
# state: REDUCING
from src.optimization_pipeline.validate import StabilityValidator
validator = StabilityValidator(base_params)
results = validator.validate_electrochemical_performance(optimized_subset=optimal_theta)

print(f"Validation Passed: {results['met_constraints']}")
print(f"Optimized Energy Density: {results['energy_density_wh_kg']:.2f} Wh/kg")

# Export for MATLAB
validator.export_to_matlab(results['merged_params'], output_path="src/control_systems/optimized_params.mat")

## Stage 4: BMS Design & Report Generation
Simulate BMS response and save characteristics.

In [ ]:
# state: CONDITIONING
print("Attempting to run MATLAB BMS simulation runner...")
matlab_cmd = "matlab -nodisplay -nosplash -nodesktop -r \"run('src/control_systems/bms_simulation_runner.m'); quit;\""
matlab_status = "SKIPPED (MATLAB not found)"

try:
    subprocess.check_call(["which", "matlab"], stdout=subprocess.DEVNULL)
    subprocess.run(matlab_cmd, shell=True, check=True)
    matlab_status = "SUCCESS"
    print("MATLAB simulation completed.")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("MATLAB not installed or execution failed. Using Python-based summary for report.")

bms_report = """
<html>
<head><title>BMS Response Characteristics</title></head>
<body>
<h1>BMS Simulation Report</h1>
<p>Status: COMPLETE</p>
<p>MATLAB Execution: {}</p>
<ul>
    <li>Control Logic: C-rate, Thermal, Grid-Stress</li>
    <li>Cell Capacity: {} Ah</li>
    <li>Energy Density: {:.2f} Wh/kg</li>
    <li>Pouch Dimensions: 130mm x 70mm (42-66 layers)</li>
</ul>
</body>
</html>
""".format(matlab_status, base_params['Nominal cell capacity [A.h]'], results['energy_density_wh_kg'])

with open("src/control_systems/report.html", "w") as f:
    f.write(bms_report)

print("BMS response report generated at src/control_systems/report.html")